<a href="https://colab.research.google.com/github/Rayen-MANSOUR/SmartDB_Explorer/blob/main/text2Sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:

!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain_huggingface langchain_community
!pip install -q streamlit cloudflared
!pip install -q datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


In [ ]:

model_id = "NumbersStation/nsql-350M"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Chargement du modèle en mode GPU automatique
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)


In [ ]:
model = prepare_model_for_kbit_training(model)

# Les modules utilisés par CodeGen/NSQL (ils contiennent Linear layers)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    )

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
ds = load_dataset("motherduckdb/duckdb-text2sql-25k", split="train[]")

print("Colonnes du dataset :", ds.column_names)
print("Exemple :", ds[0])

def format_prompt(example):
    """Construit un prompt clair pour Text2SQL"""
    return (
        f"Schema:\n{example['schema']}\n"
        f"Question:\n{example['prompt']}\n"
        f"SQL:\n{example['query']}"
    )

def tokenize_fn(example):
    text = format_prompt(example)
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


tokenized_ds = ds.map(tokenize_fn, remove_columns=ds.column_names)
print("Dataset tokenisé :", tokenized_ds)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # on fait du causal LM, pas de masquage
)


In [ ]:
training_args = TrainingArguments(
    output_dir="./nsql350m_lora_out",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",  # désactive WandB
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

In [ ]:
output_dir = "./nsql350m_text2sql_lora"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Modèle fine-tuné sauvegardé dans {output_dir}")

In [62]:
%%writefile text2sql_app.py
import os
import re
import torch
import sqlite3
import pandas as pd
import streamlit as st
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline



#  Chargement du modèle fine-tuné (NSQL-350M + LoRA)

@st.cache_resource
def load_model():
    model_id = "./nsql350m_text2sql_lora"

    st.write(f"🔧 Chargement du modèle `{model_id}` (4-bit quantisé, optimisé GPU)...")

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )

    model.config.use_cache = False

    gen_pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        temperature=0.0,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    llm = HuggingFacePipeline(pipeline=gen_pipe)
    return llm



def extract_sql(text: str) -> str:
    """Extrait proprement la requête SQL depuis le texte."""
    block = re.search(r"```sql(.*?)```", text, re.IGNORECASE | re.DOTALL)
    if block:
        candidate = block.group(1).strip()
    else:
        match = re.search(r"(SELECT\s+.+?;)", text, re.IGNORECASE | re.DOTALL)
        candidate = match.group(1).strip() if match else text.strip()
    candidate = candidate.replace("```", "").strip()
    if not candidate.endswith(";"):
        candidate += ";"
    return candidate


def fix_sql_for_sqlite(sql_query: str) -> str:
    corrections = {
        "ILIKE": "LIKE",
        "date_trunc": "strftime",
        "DATE_TRUNC": "strftime",
        "EXTRACT": "strftime",
        "`": "",
        '"': "",
    }
    for bad, good in corrections.items():
        sql_query = sql_query.replace(bad, good)
    return sql_query.strip()


def execute_sql_with_repair(sql_query: str, db_path: str):
    """Exécute la requête sur SQLite."""
    try:
        conn = sqlite3.connect(db_path)
        result = pd.read_sql_query(sql_query, conn)
        conn.close()
        return result
    except Exception as e:
        st.warning(f"⚠️ Erreur SQL initiale : {e}")
        fixed_query = fix_sql_for_sqlite(sql_query)
        st.code(fixed_query, language="sql")
        try:
            conn = sqlite3.connect(db_path)
            result = pd.read_sql_query(fixed_query, conn)
            conn.close()
            st.info("✅ Correction automatique appliquée.")
            return result
        except Exception as e2:
            st.error(f"❌ Échec même après correction : {e2}")
            return None



def generate_sql_query(question: str, db_path: str, llm) -> str:
    """Construit un prompt clair et efficace pour le modèle."""
    conn = sqlite3.connect(db_path)
    schema_data = pd.read_sql_query("SELECT name, sql FROM sqlite_master WHERE type='table';", conn)
    schema_text = "\n".join(
        f"{row['name']}({', '.join(re.findall(r'\"(.*?)\"', row['sql']))})"
        for _, row in schema_data.dropna().iterrows()
    )
    conn.close()

    prompt = f"""
    You are an expert SQLite developer.
    Generate one valid SQL query that answers the following question.

    ### Schema:
    {schema_text}

    ### Question:
    {question}

    SQL:
    """

    response = llm.invoke(prompt)
    response_text = str(response)

    st.markdown("###  Réponse brute du modèle")
    st.text_area("Texte complet retourné :", response_text, height=250)

    sql_query = extract_sql(response_text)
    if not sql_query or "select" not in sql_query.lower():
        match = re.search(r"(SELECT\s.+)", response_text, re.IGNORECASE)
        sql_query = match.group(1).strip() + ";" if match else None

    if not sql_query:
        st.error(" Aucune requête SQL valide extraite du modèle.")
        return "SELECT 'Erreur : aucune requête SQL valide générée.';"

    sql_query = fix_sql_for_sqlite(sql_query)
    st.markdown("###  Requête SQL extraite :")
    st.code(sql_query, language="sql")
    return sql_query



def main():
    st.set_page_config(page_title=" SmartDB Explorer : Assistant Text2SQL pour Bases de Données", layout="wide")
    st.title(" SmartDB Explorer : Assistant Text2SQL pour Bases de Données")
    st.markdown("Importez une base `.db` ou `.sqlite`, puis posez vos questions en langage naturel. 📊")

    # Charger le modèle
    if st.button("⚙️ Charger le modèle fine-tuné"):
        with st.spinner("Chargement du modèle (rapide)..."):
            llm = load_model()
            st.session_state["llm"] = llm
            st.success("✅ Modèle  chargé avec succès.")

    if "llm" not in st.session_state:
        st.warning(" Veuillez d’abord charger le modèle avant d’importer une base.")
        return

    llm = st.session_state["llm"]

    # Import base
    st.subheader("📂 Importer une base SQLite")
    uploaded_db = st.file_uploader("Choisissez un fichier .db ou .sqlite :", type=["db", "sqlite"])

    if uploaded_db:
        db_path = os.path.join("/content", uploaded_db.name)
        with open(db_path, "wb") as f:
            f.write(uploaded_db.getbuffer())
        st.success(f"✅ Base importée : `{uploaded_db.name}`")

        try:
            conn = sqlite3.connect(db_path)
            tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
            conn.close()
            st.markdown("### 🧾 Tables détectées :")
            st.dataframe(tables)

            question = st.text_input(
                "💬 Posez votre question :",
                placeholder="Ex : Quels sont les 5 employés les mieux payés ?"
            )

            if st.button("Générer et exécuter la requête") and question:
                with st.spinner("⏳ Génération de la requête SQL..."):
                    sql_query = generate_sql_query(question, db_path, llm)

                    st.markdown("### 🧩 Requête SQL générée :")
                    st.code(sql_query, language="sql")

                    result = execute_sql_with_repair(sql_query, db_path)
                    if result is not None and not result.empty:
                        st.success("Résultat :")
                        st.dataframe(result)
                    else:
                        st.warning("⚠️ Aucun résultat ou erreur persistante.")
        except Exception as e:
            st.error(f"❌ Erreur de lecture de la base : {e}")



if __name__ == "__main__":
    main()


Overwriting text2sql_app.py


In [64]:
!pip install -q streamlit cloudflared
!pkill streamlit || true
!pkill cloudflared || true
!nohup streamlit run text2sql_app.py --server.port 8501 > logs.txt 2>&1 &
!sleep 10 && npx cloudflared tunnel --url http://localhost:8501
